In [1]:
!pip install geopandas

In [9]:
import sys

PROJECT_ROOT = "/home/91/23016891/projects/westport-ibw"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


In [8]:

SIMULATION_DATA_DIR = "/data/westport/data"
BUILDING_FILE_PATH  = f"{SIMULATION_DATA_DIR}/exposure/buildings.gpkg"
PROPERTY_BUILDING_REFERENCE_PATH = f"{SIMULATION_DATA_DIR}/exposure/property_buildings_reference.csv"
PROPERTY_FILE_PATH       = f"{SIMULATION_DATA_DIR}/exposure/property_boundaries.gpkg"
PROPERTY_ADDRESS_FILE_PATH = f"{SIMULATION_DATA_DIR}/exposure/property_addresses.csv"
ADDRESS_ROADS_FILE_PATH  = f"{SIMULATION_DATA_DIR}/exposure/address_roads.gpkg"
MANUAL_ADDRESS_POINTS    = f"{SIMULATION_DATA_DIR}/exposure/manual_address_points.csv"
ADDRESS_POINTS_FILE_PATH = f"{SIMULATION_DATA_DIR}/exposure/address_points.gpkg"
EXCLUDED_BUILDINGS       = f"{SIMULATION_DATA_DIR}/exposure/excluded_buildings.csv"
ROAD_CADASTRAL           = f"{SIMULATION_DATA_DIR}/exposure/road_network.gpkg"
ROAD_PARCEL_SECTIONS     = f"{SIMULATION_DATA_DIR}/exposure/road_parcel_sections.csv"
SURVEYED_FLOOD_MAP_FILE_PATH = f"{SIMULATION_DATA_DIR}/flood-mapping-with-ml-main-DATA/DATA/July_2021_surveyed_flood_depth.tif"
SAMPLE_FLOOD_MAP = "/data/westport/data/flood_maps/Base0_Buller20yrARI_v2_depth_snapshots/depth_at_2000-01-02_00-00-00.tif"

In [14]:
import torch
from modules.utils.spatial_operators.operators import _get_feature_labels
import numpy as np
import shapely
from shapely.ops import unary_union
from shapely.geometry import shape
from rasterio import features
from rasterio.transform import Affine
import geopandas as gpd
import rasterio


building_data = gpd.read_file(BUILDING_FILE_PATH)

with rasterio.open(SAMPLE_FLOOD_MAP) as src:
    transform = src.transform
    raster_crs = src.crs
    nodata = np.nan
    data = src.read(1)
    
    #assign 0 to no data
    
CELL_SIZE =  abs(transform.a)
MAX_BUFFER_PCT=3
MIN_RING_CELLS=4
GROWTH_STEP_PCT=1

def _build_adaptive_rings(exposed_objects, depth, transform, 
                            raster_crs):
    all_buildings_geom = unary_union(exposed_objects.geometry.values)
    n = len(exposed_objects)
    buffer_pct = np.full(n, 1, dtype=float)
    needs_growth = np.ones(n, dtype=bool)

    final_geom = None
    final_counts = None
    final_buffer_pct = buffer_pct.copy()

    while needs_growth.any():
        dist = buffer_pct * CELL_SIZE
        candidate_geom = exposed_objects.geometry.buffer(dist).difference(all_buildings_geom)

        labels, n_feat = _get_feature_labels(
            gpd.GeoDataFrame(geometry=candidate_geom, crs=exposed_objects.crs),
            tuple(depth.shape), transform, raster_crs,
        )
        labels_gpu = torch.from_numpy(labels).long().to(depth.device)
        _, _, _, _, counts, _ = self._gpu_zonal_stats(depth, labels_gpu, n_feat)
        counts_np = counts.cpu().numpy()

        if final_geom is None:
            final_geom, final_counts = candidate_geom, counts_np
        else:
            update = needs_growth
            final_geom = final_geom.where(~update, candidate_geom)
            final_counts = np.where(update, counts_np, final_counts)

        final_buffer_pct = np.where(needs_growth, buffer_pct, final_buffer_pct)

        maxed_out = buffer_pct >= MAX_BUFFER_PCT
        needs_growth = (final_counts < MIN_RING_CELLS) & (~maxed_out)
        buffer_pct = np.where(needs_growth, buffer_pct + GROWTH_STEP_PCT, buffer_pct)

    # these need the block-dissolve fallback, not a bigger ring.
    exhausted = (final_counts < MIN_RING_CELLS) & (final_buffer_pct >=  MAX_BUFFER_PCT)
    return final_geom, final_counts, exhausted

final_geom, final_counts, exhausted = _build_adaptive_rings(building_data, data, transform, raster_crs)



            

NameError: name 'self' is not defined

In [ ]:
import sys

EXPLORATORY = f"{PROJECT_ROOT}/exploratory"
if EXPLORATORY not in sys.path:
    sys.path.insert(0, EXPLORATORY)

from ring_growth_tuning import sweep

# Scalars or lists for either parameter. The grid and the building layer are
# loaded once and kept, so re-running with different floors only pays for the
# rasterization. write=False keeps it out of out/ while tuning.
frames, summary = sweep(min_cells=[4, 6], max_buffer_pct=[3, 5], write=False)
summary


In [18]:
BUILDING_RINGS_PATH = "/home/91/23016891/projects/westport-ibw/out/building_rings.geojson"
building_rings = gpd.read_file(BUILDING_RINGS_PATH)


In [21]:
import numpy as np
import geopandas as gpd
import rasterio
from modules.utils.spatial_operators.operators import _get_feature_labels, get_sample_georeference

WORK_CRS = 2193

rings = gpd.read_file(BUILDING_RINGS_PATH).to_crs(WORK_CRS)

transform, raster_crs, _ = get_sample_georeference(SAMPLE_FLOOD_MAP)
with rasterio.open(SAMPLE_FLOOD_MAP) as src:
    shape = src.shape
cell_area = abs(transform.a * transform.e)

# allow_overlap=True and a bincount over the feature index — the same two
# operations _gpu_zonal_stats_sparse performs to produce num_ring_cells.
_, _, feat_idx, n = _get_feature_labels(
    rings, shape, transform, raster_crs, allow_overlap=True,
)
rings["num_ring_cells"] = np.bincount(feat_idx, minlength=n)
rings["ring_area_m2"] = rings.area.round(1)
rings["area_in_cells"] = (rings["ring_area_m2"] / cell_area).round(2)
